In [15]:
import json
import pandas as pd

# dataset - 1

In [33]:
with open("normalized_products/dataset_1_normalized.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [17]:
# rows, columns
print(df.shape)

(812, 27)


In [18]:
print(df.columns)

Index(['id', 'brand', 'title', 'description', 'price', 'priceCurrency',
       'cluster_id', 'url', 'title_description', 'model', 'model_number',
       'product_type', 'chipset_name', 'vram_gb', 'storage_gb',
       'read_speed_mb_s', 'write_speed_mb_s', 'bus_type', 'interface_type',
       'width_mm', 'length_mm', 'height_mm', 'weight_g',
       'storage_connection_type', 'memory_type', 'color', 'form_factor'],
      dtype='object')


In [19]:
print(df.head())

         id            brand  \
0  12198483         Gigabyte   
1  78378158  Western Digital   
2  80641070          Corsair   
3  51886539          Seagate   
4  10328354             Asus   

                                               title  \
0  Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...   
1            WD Blue 6TB 3.5\" SATA 3 HDD/Hard Drive   
2                Corsair Force MP510 M.2 SSD - 960GB   
3         Seagate EXOS 4TB 3.5\" SATA HDD/Hard Drive   
4    Asus GeForce GTX 1650 Phoenix OC 4GB Video Card   

                                         description    price priceCurrency  \
0  CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...   799.99           GBP   
1  6TB WD Blue WD60EZAZ, 3.5\" HDD, SATA III - 6G...   129.98           GBP   
2  Solid State Drive, 960 GB, intern, M.2 2280, P...  2124.00           NOK   
3  4TB Seagate EXOS ST4000NM0115, 3.5\" Enterpris...   142.99           GBP   
4  The ASUS GeForce GTX 1650 Phoenix OC 4GB Video...   219.00           AUD

In [20]:
df.isnull().sum().sort_values(ascending=False)

weight_g                   786
length_mm                  767
width_mm                   767
height_mm                  766
color                      692
write_speed_mb_s           690
read_speed_mb_s            661
memory_type                602
vram_gb                    583
chipset_name               581
model_number               478
storage_connection_type    399
form_factor                395
interface_type             376
storage_gb                 241
bus_type                   184
model                       64
priceCurrency               43
price                       40
brand                       27
product_type                 0
title_description            0
url                          0
cluster_id                   0
description                  0
title                        0
id                           0
dtype: int64

The most interesting attributes are the ones where some values exist but many are missing. Those are exactly where a RAG system can shine. Here are the ones that we shall care about: 

- model_number (478 missing → ~40% present)
- storage_connection_type (399 missing)
- form_factor (395 missing)
- interface_type (376 missing)
- storage_size (241 missing)
- bus_type (184 missing)
- vram (583 missing but still present for GPUs)

In [21]:
df[df.cluster_id == 1002037]

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,...,bus_type,interface_type,width_mm,length_mm,height_mm,weight_g,storage_connection_type,memory_type,color,form_factor
0,12198483,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",799.99,GBP,1002037,https://www.novatech.co.uk/products/gigabyte-n...,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,NVIDIA GeForce RTX 3080 Gaming OC,...,None,None,NaN,NaN,NaN,NaN,None,GDDR6X,None,None


In [22]:
cluster = df[df.cluster_id == 1002037]
cluster[["title", "vram_gb", "storage_gb", "interface_type", "bus_type", "form_factor", "storage_connection_type", "model_number"]]

,title,vram_gb,storage_gb,interface_type,bus_type,form_factor,storage_connection_type,model_number
0,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,10.0,NaN,None,None,None,None,None


In [23]:
df["cluster_id"].nunique()
df.groupby("cluster_id").size().describe()

count    812.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
dtype: float64

- That means every cluster contains only one row. Hence, each product appears only once in this file.
- So the clusters you expected (multiple offers for the same product) are not inside this single dataset.

In [24]:
def load_json(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

df1 = load_json("normalized_products/dataset_1_normalized.json")
df2 = load_json("normalized_products/dataset_2_normalized.json")
df3 = load_json("normalized_products/dataset_3_normalized.json")
df4 = load_json("normalized_products/dataset_4_normalized.json")

In [25]:
# combine them all
all_df = pd.concat([df1, df2, df3, df4])

In [26]:
print(all_df.shape)

(3012, 27)


In [27]:
# checking clusters again:
all_df.groupby("cluster_id").size().describe()

count    812.000000
mean       3.709360
std        0.574219
min        2.000000
25%        4.000000
50%        4.000000
75%        4.000000
max        4.000000
dtype: float64

- That means each product appears in about 3–4 different datasets.
- This is perfect for retrieval-based repair. One row is missing a value, while another store listing contains it.

In [28]:
# identify which attributes can actually be repaired using cluster information
attributes = [
    "model_number",
    "storage_gb",
    "bus_type",
    "interface_type",
    "form_factor",
    "vram_gb",
    "storage_connection_type"
]


for attr in attributes:
    repairable = 0
    missing = 0
    
    for cid, group in all_df.groupby("cluster_id"):
        if group[attr].notnull().any():
            missing_rows = group[group[attr].isnull()]
            repairable += len(missing_rows)
        missing += group[attr].isnull().sum()
    
    print("Attribute: ", attr, "Repairable: ", repairable, "Total missing: ", missing)

Attribute:  model_number Repairable:  1187 Total missing:  1801
Attribute:  storage_gb Repairable:  31 Total missing:  912
Attribute:  bus_type Repairable:  623 Total missing:  726
Attribute:  interface_type Repairable:  318 Total missing:  1438
Attribute:  form_factor Repairable:  276 Total missing:  1485
Attribute:  vram_gb Repairable:  17 Total missing:  2163
Attribute:  storage_connection_type Repairable:  241 Total missing:  1511


RAG approach can actually make a difference. Let’s unpack it:

- model_number → 1187 / 1801 → Huge portion of missing values can actually be retrieved from other rows in the same cluster. Excellent candidate.
- bus_type → 623 / 726 → Also very promising.
- storage_size → 31 / 912 → Only a tiny fraction can be recovered. Might not be worth spending time here.
- interface_type → 318 / 1438 → Some potential.
- form_factor → 276 / 1485 → Moderate.
- vram → 17 / 2163 → Almost nothing to gain. Skip.
- storage_connection_type → 241 / 1511 → Some potential, but low.

For your first experiments, focus on model_number and bus_type. They have enough missing-but-recoverable values to generate meaningful results.

In [32]:

import pandas as pd

df1 = pd.read_json('normalized_products/dataset_1_normalized.json')  # update paths
df2 = pd.read_json('normalized_products/dataset_2_normalized.json')
df3 = pd.read_json('normalized_products/dataset_3_normalized.json')
df4 = pd.read_json('normalized_products/dataset_4_normalized.json')
kb = pd.concat([df2, df3, df4], ignore_index=True)

target_attributes = ['model_number', 'bus_type', 'interface_type',
                     'form_factor', 'storage_connection_type', 'storage_gb', 'vram_gb']

def get_gt(cluster_id, attr, kb):
    matches = kb[kb['cluster_id'] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attr)
        if pd.notna(val) and str(val).strip().lower() not in {'', 'none', 'nan'}:
            return str(val).strip()
    return None

for attr in target_attributes:
    missing_rows = df1[df1[attr].isna()]
    recoverable = sum(1 for _, row in missing_rows.iterrows() if get_gt(row['cluster_id'], attr, kb) is not None)
    print(f'{attr}: {len(missing_rows)} missing, {recoverable} recoverable ({100*recoverable/max(len(missing_rows),1):.0f}%)')


model_number: 478 missing, 302 recoverable (63%)
bus_type: 184 missing, 154 recoverable (84%)
interface_type: 376 missing, 64 recoverable (17%)
form_factor: 395 missing, 61 recoverable (15%)
storage_connection_type: 399 missing, 46 recoverable (12%)
storage_gb: 241 missing, 6 recoverable (2%)
vram_gb: 583 missing, 2 recoverable (0%)


In [34]:
import pandas as pd

df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")
kb = pd.concat([df2, df3, df4], ignore_index=True)

def get_ground_truth(cluster_id, attribute, kb):
    matches = kb[kb["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

# Check ALL attributes Ralph mentioned + your existing ones
candidates = [
    "width_mm", "height_mm", "length_mm",
    "read_speed_mb_s", "write_speed_mb_s",
    "model_number", "model", "bus_type",
    "form_factor", "interface_type",
    "storage_connection_type", "storage_gb", "vram_gb"
]

print(f"{'Attribute':<30} {'Missing':>8} {'Recoverable':>12} {'Rate':>6}")
print("-" * 60)
for attr in candidates:
    if attr not in df1.columns:
        print(f"{attr:<30} {'NOT IN DATASET':>30}")
        continue
    missing_rows = df1[df1[attr].isna()]
    if len(missing_rows) == 0:
        print(f"{attr:<30} {'no missing values':>30}")
        continue
    recoverable = sum(
        1 for _, row in missing_rows.iterrows()
        if get_ground_truth(row["cluster_id"], attr, kb) is not None
    )
    rate = 100 * recoverable / len(missing_rows)
    print(f"{attr:<30} {len(missing_rows):>8} {recoverable:>12} {rate:>5.0f}%")

Attribute                       Missing  Recoverable   Rate
------------------------------------------------------------
width_mm                            767           84    11%
height_mm                           766          101    13%
length_mm                           767           78    10%
read_speed_mb_s                     661          122    18%
write_speed_mb_s                    690           87    13%
model_number                        478          302    63%
model                                64           47    73%
bus_type                            184          154    84%
form_factor                         395           61    15%
interface_type                      376           64    17%
storage_connection_type             399           46    12%
storage_gb                          241            6     2%
vram_gb                             583            2     0%
